# 📚 PadhaiLLM — The Syllabus Bot / Course Exam Prep Assistant

## A Local RAG (Retrieval-Augmented Generation) System

---

| Component | Technology |
|---|---|
| **LLM** | Llama 3.2 (3B) via Ollama |
| **Embeddings** | all-MiniLM-L6-v2 (Sentence Transformers) |
| **Vector Database** | ChromaDB |
| **PDF Extraction** | PyMuPDF + pdfplumber + pypdf (multi-strategy) |
| **Framework** | LangChain + Streamlit |
| **Retrieval** | MMR (Maximal Marginal Relevance) |

---

## 1. Architecture Overview

```
┌─────────────┐    ┌──────────────────┐    ┌──────────────┐    ┌─────────────┐
│  PDF Upload  │───▶│  Multi-Strategy   │───▶│  Recursive    │───▶│  ChromaDB   │
│  (Streamlit) │    │  Text Extraction  │    │  Chunking     │    │  Vectors    │
└─────────────┘    └──────────────────┘    └──────────────┘    └──────┬──────┘
                                                                      │
┌─────────────┐    ┌──────────────────┐    ┌──────────────┐           │
│  Answer to   │◀──│  Llama 3.2 (3B)  │◀──│  Prompt +     │◀──────────┘
│  Student     │    │  via Ollama      │    │  Retrieved    │   (MMR Search)
└─────────────┘    └──────────────────┘    │  Chunks       │
                                           └──────────────┘
```

## 2. Environment Setup & Imports

In [ ]:
# Core Libraries
import os
import re
import hashlib

# PDF Extraction (Multi-Strategy)
import fitz                    # PyMuPDF — primary extractor
import pdfplumber              # Fallback — better for tables
from pypdf import PdfReader    # Last resort — most compatible

# LangChain Components
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

print("All imports successful ✅")

## 3. PDF Text Extraction — The Core Fix

### Why the old code failed:
- `PyPDFLoader` uses only `pypdf` internally, which struggles with:
  - Complex layouts, multi-column text
  - Embedded fonts, ligatures
  - Tables and structured data
  - Scanned / image-heavy PDFs

### Our Solution: Multi-Strategy Extraction
1. **PyMuPDF** (primary) — fastest, best general-purpose extraction
2. **pdfplumber** (fallback) — excels at tables and complex layouts
3. **pypdf** (last resort) — most compatible, fills gaps

In [ ]:
def extract_text_pymupdf(pdf_path: str) -> list:
    """Strategy 1: PyMuPDF — fast, handles most PDFs well."""
    documents = []
    try:
        doc = fitz.open(pdf_path)
        for page_num in range(len(doc)):
            page = doc[page_num]
            text = page.get_text("text")
            # Fallback to blocks if text mode yields too little
            if len(text.strip()) < 50:
                blocks = page.get_text("blocks")
                block_texts = [b[4] for b in blocks if b[6] == 0]
                text = "\n".join(block_texts)
            if text.strip():
                documents.append(Document(
                    page_content=text.strip(),
                    metadata={"page": page_num + 1, "source": os.path.basename(pdf_path), "extractor": "pymupdf"}
                ))
        doc.close()
    except Exception as e:
        print(f"PyMuPDF warning: {e}")
    return documents


def extract_text_pdfplumber(pdf_path: str) -> list:
    """Strategy 2: pdfplumber — better for tables and complex layouts."""
    documents = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages):
                text_parts = []
                main_text = page.extract_text(x_tolerance=3, y_tolerance=3)
                if main_text:
                    text_parts.append(main_text)
                # Also extract tables
                for table in page.extract_tables():
                    table_text = "\n".join(" | ".join(str(c or "") for c in row) for row in table)
                    if table_text.strip():
                        text_parts.append(f"\n[Table]\n{table_text}\n[/Table]")
                combined = "\n".join(text_parts)
                if combined.strip():
                    documents.append(Document(
                        page_content=combined.strip(),
                        metadata={"page": page_num + 1, "source": os.path.basename(pdf_path), "extractor": "pdfplumber"}
                    ))
    except Exception as e:
        print(f"pdfplumber warning: {e}")
    return documents


def extract_text_pypdf(pdf_path: str) -> list:
    """Strategy 3: pypdf — most compatible, last resort."""
    documents = []
    try:
        reader = PdfReader(pdf_path)
        for page_num, page in enumerate(reader.pages):
            text = page.extract_text() or ""
            if text.strip():
                documents.append(Document(
                    page_content=text.strip(),
                    metadata={"page": page_num + 1, "source": os.path.basename(pdf_path), "extractor": "pypdf"}
                ))
    except Exception as e:
        print(f"pypdf warning: {e}")
    return documents


def robust_pdf_extract(pdf_path: str) -> list:
    """Master function: tries all strategies, picks best coverage."""
    docs_mupdf = extract_text_pymupdf(pdf_path)
    chars_mupdf = sum(len(d.page_content) for d in docs_mupdf)

    try:
        total_pages = len(fitz.open(pdf_path))
    except:
        total_pages = max(len(docs_mupdf), 1)

    coverage = len(docs_mupdf) / total_pages if total_pages > 0 else 0

    if coverage >= 0.8 and chars_mupdf > 100:
        print(f"✅ PyMuPDF extracted {len(docs_mupdf)}/{total_pages} pages ({chars_mupdf:,} chars)")
        return docs_mupdf

    docs_plumber = extract_text_pdfplumber(pdf_path)
    chars_plumber = sum(len(d.page_content) for d in docs_plumber)

    primary = docs_plumber if chars_plumber > chars_mupdf else docs_mupdf
    name = "pdfplumber" if chars_plumber > chars_mupdf else "PyMuPDF"

    if len(primary) / total_pages < 0.5:
        docs_pypdf = extract_text_pypdf(pdf_path)
        extracted = {d.metadata["page"] for d in primary}
        for doc in docs_pypdf:
            if doc.metadata["page"] not in extracted:
                primary.append(doc)
        primary.sort(key=lambda d: d.metadata["page"])

    chars = sum(len(d.page_content) for d in primary)
    print(f"✅ {name} extracted {len(primary)}/{total_pages} pages ({chars:,} chars)")
    return primary


def clean_text(text: str) -> str:
    """Fix common extraction artifacts."""
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'(\w)-\n(\w)', r'\1\2', text)
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', text)
    return text.strip()

print("Extraction functions defined ✅")

## 4. Test PDF Extraction

Upload a PDF to `test_doc.pdf` in the project directory to test extraction.

In [ ]:
# Change this path to test with your PDF
TEST_PDF = "test_doc.pdf"

if os.path.exists(TEST_PDF):
    docs = robust_pdf_extract(TEST_PDF)
    for doc in docs:
        doc.page_content = clean_text(doc.page_content)
    docs = [d for d in docs if len(d.page_content) > 20]

    print(f"\nTotal documents: {len(docs)}")
    print(f"Total characters: {sum(len(d.page_content) for d in docs):,}")
    print(f"\n--- Page 1 Preview (first 500 chars) ---")
    if docs:
        print(docs[0].page_content[:500])
else:
    print(f"Place a PDF at '{TEST_PDF}' to test extraction.")

## 5. Chunking Strategy

### Old vs New:
| Parameter | Old | New | Why |
|---|---|---|---|
| `chunk_size` | 600 | 1000 | Larger chunks = more context per retrieval |
| `chunk_overlap` | 60 | 200 | Prevents information loss at chunk boundaries |
| `separators` | default | custom | Academic text has specific structure |

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", "; ", ", ", " ", ""],
    length_function=len,
)

if os.path.exists(TEST_PDF) and docs:
    chunks = text_splitter.split_documents(docs)
    print(f"Documents: {len(docs)} → Chunks: {len(chunks)}")
    print(f"Avg chunk size: {sum(len(c.page_content) for c in chunks) // len(chunks)} chars")
    print(f"\n--- Sample Chunk ---")
    print(chunks[0].page_content[:300])
else:
    print("No test PDF loaded. Skipping chunking demo.")

## 6. Embedding & Vector Store

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
)

# Test embedding
test_embed = embeddings.embed_query("What is machine learning?")
print(f"Embedding dimension: {len(test_embed)}")
print(f"Sample values: {test_embed[:5]}")

In [ ]:
if os.path.exists(TEST_PDF) and docs:
    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name="notebook_test",
    )

    retriever = vector_store.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 8, "fetch_k": 20, "lambda_mult": 0.7}
    )

    print(f"Vector store created with {len(chunks)} vectors ✅")
    print(f"Retriever: MMR search, k=8, fetch_k=20")
else:
    print("No test PDF loaded. Skipping vector store creation.")

## 7. LLM Configuration

In [ ]:
llm = ChatOllama(
    model="llama3.2:3b",
    temperature=0,          # Deterministic output
    num_ctx=8192,           # Extended context window
    top_p=0.1,              # Restrict to high-probability tokens
    top_k=20,               # Limit vocabulary pool
    repeat_penalty=1.15,    # Prevent repetition loops
)

# Quick test
response = llm.invoke("Say 'PadhaiLLM is ready' and nothing else.")
print(f"LLM Response: {response.content}")

## 8. Prompt Engineering & Chain Assembly

In [ ]:
system_prompt = (
    "You are **PadhaiLLM**, a strict, highly accurate academic assistant.\n\n"
    "## RULES\n"
    "1. Answer using **ONLY** the provided context chunks.\n"
    "2. Use bullet points and bold text for clarity.\n"
    "3. If the answer is NOT in the context, say so explicitly.\n"
    "4. NEVER generate from your own training data.\n\n"
    "## CONTEXT CHUNKS\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

print("Prompt template configured ✅")

In [ ]:
if os.path.exists(TEST_PDF) and docs:
    combine_docs_chain = create_stuff_documents_chain(llm, prompt)
    rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

    # Test query
    test_query = "Summarize the main topics covered in this document."
    print(f"Query: {test_query}\n")
    result = rag_chain.invoke({"input": test_query})
    print(f"Answer:\n{result['answer']}")
    print(f"\nSources used: {len(result.get('context', []))} chunks")
else:
    print("No test PDF loaded. Skipping RAG chain test.")

## 9. Running the Streamlit App

The complete production app is in `app.py`. Run it with:

```bash
streamlit run app.py
```

### Key Improvements over the original:

| Feature | Before | After |
|---|---|---|
| **PDF Extraction** | `PyPDFLoader` only | PyMuPDF + pdfplumber + pypdf (3 strategies) |
| **Chunk Size** | 600 chars | 1000 chars |
| **Chunk Overlap** | 60 chars | 200 chars |
| **Retrieval** | Similarity (k=6) | MMR (k=8, diverse results) |
| **Cache** | `@st.cache_resource` (stale) | Hash-based invalidation |
| **Tables** | Not extracted | Extracted via pdfplumber |
| **UI** | Basic dark theme | Premium glassmorphism, animations |

---

*PadhaiLLM — Built for exam preparation, powered by local AI.*